# LSE News Explorer — Scraper (Módulo 1, tracker retail UK)

Trae comunicados del London Stock Exchange y los filtra por sector **después** de bajarlos — sin AI, sin tokens, solo `requests`.

**Cambio de estrategia vs. la primera versión**: el filtro de sector del endpoint de listado (`sector=...`) no funciona — el LSE lo ignora silenciosamente. En vez de seguir adivinando el nombre del parámetro, bajamos el detalle de cada comunicado de la ventana (endpoint separado) y filtramos con la clasificación ICB oficial que viene ahí (`icbsectorcode`). De paso, ese mismo request trae el `body` completo del comunicado — dos problemas resueltos con una sola llamada extra por artículo.

**Config separada**: este notebook lee `lse_config.json` (local → GitHub raw → default embebido, en ese orden).

## Setup

In [ ]:
import html as html_lib
import json
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 80)

## Config (local → GitHub → default embebido)

In [ ]:
CONFIG_PATH = Path("lse_config.json")

GITHUB_CONFIG_URL = (
    "https://raw.githubusercontent.com/FUranga/retail-news-tracker/main/lse_config.json"
)

DEFAULT_CONFIG = {
    "endpoint": {
        "base_url": "https://api.londonstockexchange.com/api/v1/components/refresh",
        "component_id": "block_content:431d02ac-09b8-40c9-aba6-04a72a4f2e49",
        "tab_id": "58734a12-d97c-40cb-8047-df76e660f23f",
    },
    "filters": {"sectors": ["404010", "452010"]},
    "company_overrides": {"include": [], "exclude": []},
    "request": {"page_size": 50, "max_pages": 20, "sleep_seconds": 0.5},
}


def load_config(path=CONFIG_PATH, github_url=GITHUB_CONFIG_URL):
    if path.exists():
        print(f"[INFO] Usando config local: {path}")
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    if github_url:
        try:
            resp = requests.get(github_url, timeout=10)
            if resp.status_code == 200:
                print(f"[INFO] Usando config desde GitHub: {github_url}")
                return resp.json()
            print(f"[WARN] GitHub respondió {resp.status_code} para {github_url}")
        except Exception as e:
            print(f"[WARN] No se pudo traer config de GitHub: {e}")
    print("[INFO] Usando DEFAULT_CONFIG embebido (ni local ni GitHub disponibles).")
    return DEFAULT_CONFIG


config = load_config()
config

## Headers y endpoints

In [ ]:
LIST_URL = "https://api.londonstockexchange.com/api/v1/components/refresh"
DETAIL_URL = "https://api.londonstockexchange.com/api/v1/pages"

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "content-type": "application/json",
    "origin": "https://www.londonstockexchange.com",
    "referer": "https://www.londonstockexchange.com/",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    ),
}

## Funciones: URL reconstruction + limpieza de HTML

In [ ]:
def slugify(title: str) -> str:
    s = (title or "").strip().lower()
    s = re.sub(r"[^a-z0-9\s-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = re.sub(r"-+", "-", s)
    return s.strip("-")


def build_article_url(companycode, title, article_id):
    if not companycode or not article_id:
        return None
    slug = slugify(title)
    return f"https://www.londonstockexchange.com/news-article/{companycode}/{slug}/{article_id}"


def strip_html(raw_html):
    """Convierte el body HTML del comunicado en texto plano legible."""
    if not raw_html:
        return ""
    text = re.sub(r"<[^>]+>", " ", raw_html)
    text = html_lib.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## Funciones: listado (SIN filtro de sector — no funciona en este endpoint)

In [ ]:
def fetch_list_page(config, page=0):
    endpoint = config["endpoint"]
    size = config["request"]["page_size"]

    top_params = f"tab=news-explorer&tabId={endpoint['tab_id']}"
    component_params = f"page={page}&size={size}&sort=datetime,desc"

    body = {
        "path": "news",
        "parameters": top_params,
        "components": [
            {"componentId": endpoint["component_id"], "parameters": component_params}
        ],
    }
    resp = requests.post(LIST_URL, headers=HEADERS, json=body, timeout=15)
    resp.raise_for_status()
    return resp.json()


def _find_search_block(payload):
    for block in payload:
        for item in block.get("content", []):
            if item.get("name") == "newsexplorersearch" and item.get("value"):
                return item["value"]
    return None


def fetch_all_list(config):
    all_articles = []
    page = 0
    max_pages = config["request"]["max_pages"]
    sleep_seconds = config["request"]["sleep_seconds"]

    while page < max_pages:
        payload = fetch_list_page(config, page=page)
        block = _find_search_block(payload)
        if not block:
            print(f"[WARN] No se encontró bloque de resultados en page={page}")
            break

        all_articles.extend(block.get("content", []))

        total_pages = block.get("totalPages", 1)
        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(sleep_seconds)

    return all_articles


def to_dataframe(articles):
    rows = []
    for a in articles:
        rows.append(
            {
                "id": a.get("id"),
                "datetime": a.get("datetime"),
                "title": a.get("title"),
                "companyname": a.get("companyname"),
                "companycode": a.get("companycode"),
                "category": a.get("category"),
                "source": a.get("source"),
                "rnsnumber": a.get("rnsnumber"),
                "url": build_article_url(a.get("companycode"), a.get("title"), a.get("id")),
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.sort_values("datetime", ascending=False).reset_index(drop=True)
    return df

## Funciones: detalle por artículo — body real + sector ICB oficial

Acá está el cambio clave: en vez de confiar en un filtro de sector que no anda, bajamos el detalle real de cada comunicado y comparamos su `icbsectorcode` contra la config. Mismo request nos da el texto completo.

In [ ]:
def fetch_article_detail(article_id):
    params = {"path": "news-article", "parameters": f"newsId={article_id}"}
    resp = requests.get(DETAIL_URL, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def parse_article_detail(payload):
    """Busca en todos los bloques porque newsarticle e issuerreferencedata
    pueden venir en bloques distintos de la respuesta."""
    body_text = None
    icb_sector_code = None
    icb_sector = None
    for block in payload.get("components", []):
        for item in (block.get("content") or []):
            name = item.get("name")
            value = item.get("value")
            if not value:
                continue
            if name == "newsarticle":
                body_text = strip_html(value.get("body"))
            elif name == "issuerreferencedata":
                icb_sector_code = value.get("icbsectorcode")
                icb_sector = value.get("icbsector")
    return body_text, icb_sector_code, icb_sector


def enrich_and_filter(df, config):
    sectors_wanted = set(config["filters"]["sectors"])
    include_tickers = set(config["company_overrides"].get("include", []))
    exclude_tickers = set(config["company_overrides"].get("exclude", []))
    sleep_seconds = config["request"]["sleep_seconds"]

    kept_rows = []
    for _, row in df.iterrows():
        if row.get("companycode") in exclude_tickers:
            continue

        try:
            detail = fetch_article_detail(row["id"])
            body_text, icb_code, icb_sector = parse_article_detail(detail)
        except Exception as e:
            print(f"[WARN] No se pudo bajar detalle de id={row['id']}: {e}")
            body_text, icb_code, icb_sector = None, None, None

        is_wanted_sector = icb_code in sectors_wanted
        is_included_ticker = row.get("companycode") in include_tickers

        if is_wanted_sector or is_included_ticker:
            new_row = row.to_dict()
            new_row["body"] = body_text
            new_row["icbsectorcode"] = icb_code
            new_row["icbsector"] = icb_sector
            kept_rows.append(new_row)

        time.sleep(sleep_seconds)

    return pd.DataFrame(kept_rows)

## Ejecutar

In [ ]:
articles = fetch_all_list(config)
df_all = to_dataframe(articles)
print(f"{len(df_all)} artículos en la ventana (sin filtrar todavía)")

df = enrich_and_filter(df_all, config)
print(f"{len(df)} artículos retail/grocery confirmados (icbsectorcode + overrides)")
df[["datetime", "companyname", "icbsector", "title", "url"]].head(20)

## Guardar

Guarda un CSV con timestamp, incluyendo el body completo y el sector ICB confirmado — este es el archivo que alimenta el registro histórico y el dashboard.

In [ ]:
if not df.empty:
    fname = f"lse_retail_news_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(fname, index=False)
    print(f"Guardado en {fname}")